# Direct Web Research

Call a search function directly, inspect its results, then ask an agent to summarize them.

## 1. Install dependencies

In [1]:
%pip install -q openai-agents ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 36.2 MB/s eta 0:00:00


## 2. Choose a model provider

Set `PROVIDER` to `"openai"` or `"gemini"`, then add the corresponding API key to Colab Secrets.

In [9]:
import os

from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from google.colab import userdata
from openai import AsyncOpenAI

PROVIDER = "gemini"  # Change to "gemini" to use Gemini.
OPENAI_MODEL_NAME = "gpt-5-mini"
GEMINI_MODEL_NAME = "gemini-3.8-flash"

if PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    if not os.environ["OPENAI_API_KEY"]:
        raise ValueError("Add OPENAI_API_KEY to Colab Secrets.")
    model = OPENAI_MODEL_NAME
elif PROVIDER == "gemini":
    GEMINI_API_KEY = userdata.get("GeminiAPIKey")
    if not GEMINI_API_KEY:
        raise ValueError("Add GEMINI_API_KEY to Colab Secrets.")
    set_tracing_disabled(disabled=True)
    model = OpenAIChatCompletionsModel(
        model=GEMINI_MODEL_NAME,
        openai_client=AsyncOpenAI(
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key=GEMINI_API_KEY,
        ),
    )
else:
    raise ValueError("PROVIDER must be 'openai' or 'gemini'.")

## 3. Define and call the search function

In [10]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException


def search_web(query: str) -> str:
    """Search the web and return source titles, summaries, and URLs."""
    try:
        results = DDGS(timeout=15).text(query, max_results=5)
    except DDGSException as error:
        print(f"Search failed: {error}")
        return "Search is temporarily unavailable."

    print(f"Search query: {query}")
    print(f"Results found: {len(results)}")
    for index, result in enumerate(results, start=1):
        print(f"\n{index}. {result.get('title', 'Untitled source')}")
        print(result.get('body', 'No summary available.'))
        print(result.get('href', 'No URL available.'))

    return "\n\n".join(
        f"{result.get('title', 'Untitled source')}\n{result.get('body', '')}\n{result.get('href', '')}"
        for result in results
    )


question = "What is agentic AI, and how is it used in business?"
search_results = search_web(question)

Search query: What is agentic AI, and how is it used in business?
Results found: 5

1. Agentic AI, explained | MIT Sloan
February 23, 2026 - The age of agentic AI — systems that are semi- or fully autonomous and can act on their own — has arrived. Here’s what you need to know, according to MIT experts.
https://mitsloan.mit.edu/ideas-made-to-matter/agentic-ai-explained

2. What is Agentic AI? - Agentic AI Explained - AWS
3 weeks ago - Find out what is Agentic AI, how and why businesses use it, and how to use Agnetic AI on AWS.
https://aws.amazon.com/what-is/agentic-ai/

3. What is Agentic AI? | IBM
August 11, 2026 - Agentic AI is an artificial intelligence system that can accomplish a specific goal with limited supervision. It consists of ai agents—machine learning models that mimic human decision-making to solve problems in real time.
https://www.ibm.com/think/topics/agentic-ai

4. What Is Agentic AI and How Does It Work in Enterprises? | Bain & Company
May 28, 2026 - Agentic AI is the

## 4. Summarize the search results

In [11]:
from agents import Agent, Runner

agent = Agent(
    name="Research Assistant",
    instructions="Summarize only the supplied search results. End with a Sources section containing the URLs you used.",
    model=model,
)

result = await Runner.run(
    starting_agent=agent,
    input=f"Question:\n{question}\n\nSearch results:\n{search_results}",
)

print(result.final_output)

**What is Agentic AI?**
Agentic AI refers to artificial intelligence systems composed of autonomous machine learning models (AI agents) that can act on their own with semi- or full autonomy. Key capabilities include:
* Accomplishing specific goals with limited human supervision.
* Mimicking human decision-making to solve problems in real time.
* Analyzing data, setting goals, planning, acting, and adapting dynamically.

**How is it used in business?**
In an enterprise context, agentic AI is used to:
* Plan, take action, and adapt across enterprise workflows to transform business operations.
* Address and solve real-time business problems with limited oversight.

### Sources
* https://mitsloan.mit.edu/ideas-made-to-matter/agentic-ai-explained
* https://www.ibm.com/think/topics/agentic-ai
* https://www.bain.com/insights/what-is-agentic-ai-and-how-does-it-work-in-enterprises/
* https://www.uipath.com/ai/agentic-ai
